# Advanced Problems with Solutions: Copying Sequences in Python

This notebook contains advanced, practice-focused problems on shallow copies, deep copies, sequence copying techniques, object identity, nested mutable objects, tuples, strings, and custom classes.

Each problem includes:

- a clear prompt,
- runnable code,
- checks or experiments,
- and a detailed solution explanation.


## Setup

In [1]:
import copy
import sys
from pprint import pprint

## Problem 1 — Identify Which Copying Techniques Create a New Outer Object

**Difficulty:** Advanced

For each expression below, predict whether the copied object is the same object as the original.

```python
l1 = [1, 2, 3]
t1 = (1, 2, 3)
s1 = 'python'

a = l1.copy()
b = list(l1)
c = l1[:]
d = copy.copy(l1)

e = tuple(t1)
f = t1[:]
g = copy.copy(t1)

h = str(s1)
i = s1[:]
j = copy.copy(s1)
```

For each variable `a` through `j`, decide whether `is` comparison with the original returns `True` or `False`.

In [2]:
l1 = [1, 2, 3]
t1 = (1, 2, 3)
s1 = 'python'

results = {
    'l1.copy()': l1.copy() is l1,
    'list(l1)': list(l1) is l1,
    'l1[:]': l1[:] is l1,
    'copy.copy(l1)': copy.copy(l1) is l1,
    'tuple(t1)': tuple(t1) is t1,
    't1[:]': t1[:] is t1,
    'copy.copy(t1)': copy.copy(t1) is t1,
    'str(s1)': str(s1) is s1,
    's1[:]': s1[:] is s1,
    'copy.copy(s1)': copy.copy(s1) is s1,
}

pprint(results)

{'copy.copy(l1)': False,
 'copy.copy(s1)': True,
 'copy.copy(t1)': True,
 'l1.copy()': False,
 'l1[:]': False,
 'list(l1)': False,
 's1[:]': True,
 'str(s1)': True,
 't1[:]': True,
 'tuple(t1)': True}


### Solution 1

For lists, the common copying techniques create a new list object:

- `l1.copy()` returns a new list.
- `list(l1)` returns a new list.
- `l1[:]` returns a new list.
- `copy.copy(l1)` returns a new list.

So all list comparisons using `is l1` are `False`.

For tuples, copying often returns the same object:

- `tuple(t1)` returns `t1` itself when `t1` is already a tuple.
- `t1[:]` returns the original tuple when the slice is the whole tuple.
- `copy.copy(t1)` returns the same tuple because tuples are immutable.

So these tuple comparisons are usually `True`.

For strings, similar logic applies because strings are immutable:

- `str(s1)` returns `s1` itself.
- `s1[:]` returns `s1` itself.
- `copy.copy(s1)` returns `s1` itself.

**Key idea:** immutable objects do not need a new copy when the supposed copy would be indistinguishable from the original.

## Problem 2 — Shallow Copy Aliasing in Nested Lists

**Difficulty:** Advanced

Predict the output of the following code:

```python
v1 = [0, 0]
v2 = [1, 1]
line1 = [v1, v2]
line2 = line1.copy()

line2[0][0] = 99

print(line1)
print(line2)
print(line1 is line2)
print(line1[0] is line2[0])
```

Explain exactly which objects are copied and which objects are shared.

In [3]:
v1 = [0, 0]
v2 = [1, 1]
line1 = [v1, v2]
line2 = line1.copy()

line2[0][0] = 99

print(line1)
print(line2)
print(line1 is line2)
print(line1[0] is line2[0])

[[99, 0], [1, 1]]
[[99, 0], [1, 1]]
False
True


### Solution 2

`line1.copy()` creates a new outer list, so `line1 is line2` is `False`.

However, this is only a shallow copy. The new list contains references to the same nested lists `v1` and `v2`.

Therefore:

- `line1[0] is line2[0]` is `True`.
- Mutating `line2[0][0]` mutates the same inner list also visible through `line1[0]`.
- Both `line1` and `line2` show `[[99, 0], [1, 1]]`.

**Important distinction:** shallow copying duplicates the container, not the objects inside it.

## Problem 3 — Build a Two-Level Copy Manually

**Difficulty:** Advanced

You are given a list of points, where each point is itself a mutable list:

```python
line1 = [[0, 0], [10, 10], [20, 20]]
```

Write a copy expression that creates:

1. a new outer list,
2. new inner point lists,
3. but does not use `copy.deepcopy`.

Then prove that mutating an inner point in the copy does not affect the original.

In [4]:
line1 = [[0, 0], [10, 10], [20, 20]]

line2 = [point[:] for point in line1]

line2[0][0] = 999

print('line1:', line1)
print('line2:', line2)
print('outer same?', line1 is line2)
print('first inner same?', line1[0] is line2[0])

line1: [[0, 0], [10, 10], [20, 20]]
line2: [[999, 0], [10, 10], [20, 20]]
outer same? False
first inner same? False


### Solution 3

The expression:

```python
line2 = [point[:] for point in line1]
```

creates a new outer list using a list comprehension. For every inner point list, `point[:]` creates a new inner list.

So:

- `line1 is line2` is `False`.
- `line1[0] is line2[0]` is `False`.
- Mutating `line2[0][0]` does not affect `line1[0][0]`.

This is deeper than a simple shallow copy, but it is not a fully general deep copy. It only copies two levels.

## Problem 4 — Why Two-Level Copying Fails for Deeper Structures

**Difficulty:** Advanced

The following structure represents a 3D shape:

```python
shape1 = [
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]]
]
```

A developer copies it using:

```python
shape2 = [layer[:] for layer in shape1]
```

Then they mutate:

```python
shape2[0][0][0] = 999
```

Predict whether `shape1` changes. Explain why.

In [5]:
shape1 = [
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]]
]

shape2 = [layer[:] for layer in shape1]
shape2[0][0][0] = 999

print('shape1:', shape1)
print('shape2:', shape2)
print('outer same?', shape1 is shape2)
print('layer same?', shape1[0] is shape2[0])
print('point same?', shape1[0][0] is shape2[0][0])

shape1: [[[999, 2], [3, 4]], [[5, 6], [7, 8]]]
shape2: [[[999, 2], [3, 4]], [[5, 6], [7, 8]]]
outer same? False
layer same? False
point same? True


### Solution 4

`shape1` changes.

The expression `[layer[:] for layer in shape1]` creates:

- a new outer list,
- new layer lists,
- but not new innermost point lists.

So `shape1[0][0]` and `shape2[0][0]` still refer to the same list.

When `shape2[0][0][0] = 999` runs, it mutates a shared inner list.

**Lesson:** manual shallow copying at each level only works if you copy every mutable level you care about. For arbitrary nesting, use `copy.deepcopy` or a carefully designed recursive copy.

## Problem 5 — Use `deepcopy` and Verify Object Independence

**Difficulty:** Advanced

Use `copy.deepcopy` on the `shape1` structure from the previous problem.

Then verify:

1. the outer lists are different objects,
2. the middle lists are different objects,
3. the inner lists are different objects,
4. mutating the copy does not affect the original.

In [6]:
shape1 = [
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]]
]

shape2 = copy.deepcopy(shape1)
shape2[0][0][0] = 999

checks = {
    'outer objects differ': shape1 is not shape2,
    'middle objects differ': shape1[0] is not shape2[0],
    'inner objects differ': shape1[0][0] is not shape2[0][0],
    'original unchanged': shape1[0][0][0] == 1,
}

print('shape1:', shape1)
print('shape2:', shape2)
pprint(checks)

shape1: [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
shape2: [[[999, 2], [3, 4]], [[5, 6], [7, 8]]]
{'inner objects differ': True,
 'middle objects differ': True,
 'original unchanged': True,
 'outer objects differ': True}


### Solution 5

`copy.deepcopy` recursively copies the full object graph where appropriate.

For this nested list structure, it creates independent lists at every level. After mutation:

- `shape2[0][0][0]` becomes `999`.
- `shape1[0][0][0]` remains `1`.

This demonstrates the main difference between shallow copying and deep copying:

- A shallow copy copies one container level.
- A deep copy recursively copies nested mutable objects.

## Problem 6 — Shallow vs Deep Copy with Custom Classes

**Difficulty:** Advanced

Consider two classes, `Point` and `Line`.

Create a `Line` object containing two `Point` objects. Then compare the behavior of `copy.copy` and `copy.deepcopy`.

Answer:

1. Does `copy.copy(line)` create a new `Line` object?
2. Does it create new `Point` objects?
3. Does `copy.deepcopy(line)` create new `Point` objects?
4. What happens if you mutate a point inside the shallow copy?

In [7]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point({self.x}, {self.y})'


class Line:
    def __init__(self, p1, p2):
        self.p1 = p1
        self.p2 = p2

    def __repr__(self):
        return f'Line({self.p1}, {self.p2})'


line1 = Line(Point(0, 0), Point(10, 10))
line_shallow = copy.copy(line1)
line_deep = copy.deepcopy(line1)

print('line1:', line1)
print('line_shallow:', line_shallow)
print('line_deep:', line_deep)

print('\nIdentity checks:')
print('line1 is line_shallow:', line1 is line_shallow)
print('line1.p1 is line_shallow.p1:', line1.p1 is line_shallow.p1)
print('line1.p1 is line_deep.p1:', line1.p1 is line_deep.p1)

line_shallow.p1.x = 999

print('\nAfter mutating line_shallow.p1.x:')
print('line1:', line1)
print('line_shallow:', line_shallow)
print('line_deep:', line_deep)

line1: Line(Point(0, 0), Point(10, 10))
line_shallow: Line(Point(0, 0), Point(10, 10))
line_deep: Line(Point(0, 0), Point(10, 10))

Identity checks:
line1 is line_shallow: False
line1.p1 is line_shallow.p1: True
line1.p1 is line_deep.p1: False

After mutating line_shallow.p1.x:
line1: Line(Point(999, 0), Point(10, 10))
line_shallow: Line(Point(999, 0), Point(10, 10))
line_deep: Line(Point(0, 0), Point(10, 10))


### Solution 6

`copy.copy(line1)` creates a new `Line` object, so `line1 is line_shallow` is `False`.

But it does not recursively copy the `Point` objects. The shallow copied `Line` still references the same `p1` and `p2` objects as `line1`.

Therefore:

- `line1.p1 is line_shallow.p1` is `True`.
- Mutating `line_shallow.p1.x` also changes `line1.p1.x`.

`copy.deepcopy(line1)` creates a new `Line` and new `Point` objects, so:

- `line1.p1 is line_deep.p1` is `False`.
- Mutating the shallow copy does not affect the deep copy.

**Key idea:** shallow copying custom objects copies the outer object but preserves references to nested attributes.

## Problem 7 — Copying Tuples That Contain Mutable Objects

**Difficulty:** Advanced

Analyze this code:

```python
t1 = ([1, 2], [3, 4])
t2 = tuple(t1)
t3 = copy.copy(t1)
t4 = copy.deepcopy(t1)

t1[0].append(99)
```

Answer:

1. Which tuples are the same object as `t1`?
2. Which tuples observe the mutation?
3. Why is this behavior not a contradiction of tuple immutability?

In [8]:
t1 = ([1, 2], [3, 4])
t2 = tuple(t1)
t3 = copy.copy(t1)
t4 = copy.deepcopy(t1)

t1[0].append(99)

print('t1:', t1)
print('t2:', t2)
print('t3:', t3)
print('t4:', t4)

print('\nIdentity checks:')
print('t1 is t2:', t1 is t2)
print('t1 is t3:', t1 is t3)
print('t1 is t4:', t1 is t4)
print('t1[0] is t4[0]:', t1[0] is t4[0])

t1: ([1, 2, 99], [3, 4])
t2: ([1, 2, 99], [3, 4])
t3: ([1, 2, 99], [3, 4])
t4: ([1, 2], [3, 4])

Identity checks:
t1 is t2: True
t1 is t3: True
t1 is t4: False
t1[0] is t4[0]: False


### Solution 7

`tuple(t1)` returns the original tuple when `t1` is already a tuple, so `t1 is t2` is `True`.

`copy.copy(t1)` also returns the original tuple, so `t1 is t3` is `True`.

`copy.deepcopy(t1)` returns a distinct tuple containing deep-copied inner lists, so `t1 is t4` is `False`.

After `t1[0].append(99)`, `t2` and `t3` observe the change because they are the same tuple object as `t1`. Even if they were separate shallow tuple copies, they would still reference the same inner lists.

`t4` does not observe the change because its inner lists were deep-copied.

This does not contradict tuple immutability. The tuple's references are fixed, but the objects referenced by those fixed references may still be mutable.

## Problem 8 — Debug a Function with a Hidden Aliasing Bug

**Difficulty:** Advanced

The following function is intended to normalize a list of 2D points without changing the caller's original list:

```python
def normalize_points(points):
    result = points.copy()
    for point in result:
        point[0] = point[0] / 100
        point[1] = point[1] / 100
    return result
```

But it mutates the caller's nested point lists.

1. Explain the bug.
2. Fix the function without using `deepcopy`.
3. Fix the function using `deepcopy`.
4. Which version is preferable for simple 2D numeric points?

In [9]:
def normalize_points_buggy(points):
    result = points.copy()
    for point in result:
        point[0] = point[0] / 100
        point[1] = point[1] / 100
    return result


def normalize_points_manual(points):
    result = [point[:] for point in points]
    for point in result:
        point[0] = point[0] / 100
        point[1] = point[1] / 100
    return result


def normalize_points_deepcopy(points):
    result = copy.deepcopy(points)
    for point in result:
        point[0] = point[0] / 100
        point[1] = point[1] / 100
    return result


original = [[10, 20], [30, 40]]
buggy = normalize_points_buggy(original)
print('After buggy call:')
print('original:', original)
print('buggy:', buggy)

original = [[10, 20], [30, 40]]
manual = normalize_points_manual(original)
print('\nAfter manual fixed call:')
print('original:', original)
print('manual:', manual)

original = [[10, 20], [30, 40]]
deep = normalize_points_deepcopy(original)
print('\nAfter deepcopy fixed call:')
print('original:', original)
print('deep:', deep)

After buggy call:
original: [[0.1, 0.2], [0.3, 0.4]]
buggy: [[0.1, 0.2], [0.3, 0.4]]

After manual fixed call:
original: [[10, 20], [30, 40]]
manual: [[0.1, 0.2], [0.3, 0.4]]

After deepcopy fixed call:
original: [[10, 20], [30, 40]]
deep: [[0.1, 0.2], [0.3, 0.4]]


### Solution 8

The bug is that `points.copy()` only copies the outer list. The inner point lists are still shared.

When the function modifies `point[0]` and `point[1]`, it mutates the same inner lists owned by the caller.

The manual fix is:

```python
result = [point[:] for point in points]
```

This is appropriate because the structure is known: a list of 2D point lists.

The `deepcopy` fix is more general:

```python
result = copy.deepcopy(points)
```

For simple 2D numeric points, the manual version is usually preferable because it is explicit, avoids unnecessary generality, and communicates the expected structure clearly.

For arbitrary nested data, `deepcopy` is safer.

## Problem 9 — Implement a Recursive Copy for Lists Only

**Difficulty:** Advanced

Implement `deep_copy_lists_only(obj)` that recursively copies lists but leaves non-list objects unchanged.

Then compare it with `copy.deepcopy` on a nested list of immutable integers.

Finally, explain why this is not a full replacement for `copy.deepcopy`.

In [10]:
def deep_copy_lists_only(obj):
    if isinstance(obj, list):
        return [deep_copy_lists_only(item) for item in obj]
    return obj


nested = [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
custom = deep_copy_lists_only(nested)
standard = copy.deepcopy(nested)

custom[0][0][0] = 999

print('nested:', nested)
print('custom:', custom)
print('standard:', standard)

print('\nIdentity checks:')
print('nested is custom:', nested is custom)
print('nested[0] is custom[0]:', nested[0] is custom[0])
print('nested[0][0] is custom[0][0]:', nested[0][0] is custom[0][0])

nested: [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]
custom: [[[999, 2], [3, 4]], [[5, 6], [7, 8]]]
standard: [[[1, 2], [3, 4]], [[5, 6], [7, 8]]]

Identity checks:
nested is custom: False
nested[0] is custom[0]: False
nested[0][0] is custom[0][0]: False


### Solution 9

`deep_copy_lists_only` recursively copies every list it finds. For nested lists of immutable values like integers, this behaves similarly to `copy.deepcopy`.

However, it is not a full replacement for `copy.deepcopy` because it does not handle:

- dictionaries,
- sets,
- tuples containing mutable elements,
- custom classes,
- shared references that should remain shared,
- circular references.

`copy.deepcopy` has machinery for memoization and object-specific copy behavior. A simple recursive function is fine for controlled structures, but risky for general-purpose copying.

## Problem 10 — Circular References and Why `deepcopy` Needs Memoization

**Difficulty:** Expert

Create a list that contains itself:

```python
a = []
a.append(a)
```

Then deep-copy it.

Answer:

1. Why would a naive recursive deep copy fail?
2. How does `copy.deepcopy` handle this case?
3. What identity relationship should hold in the copied object?

In [11]:
a = []
a.append(a)

b = copy.deepcopy(a)

print('a is a[0]:', a is a[0])
print('b is b[0]:', b is b[0])
print('a is b:', a is b)

a is a[0]: True
b is b[0]: True
a is b: False


### Solution 10

A naive recursive copy would keep trying to copy the list inside itself forever:

```text
copy a -> copy a[0] -> copy a[0][0] -> ...
```

This would eventually cause infinite recursion or a recursion depth error.

`copy.deepcopy` avoids this by using a memo dictionary internally. Once it starts copying an object, it remembers the original object's identity and the copy being built. If the same original object is encountered again, `deepcopy` reuses the already-created copy instead of recursively copying again.

Therefore:

- `a is a[0]` is `True`.
- `b is b[0]` is also `True`.
- `a is b` is `False`.

The copied structure preserves the self-reference, but it is a distinct object graph.

## Final Review Checklist

After completing these problems, you should be able to explain:

- why list copies create new outer list objects,
- why tuple and string copies often return the original object,
- what shallow copy means,
- how nested mutable objects create aliasing bugs,
- when manual nested copying is sufficient,
- when `copy.deepcopy` is safer,
- how custom classes behave under shallow and deep copy,
- and why circular references require memoization.